In [1]:
import os
import random
from pathlib import Path
from PIL import Image, ImageEnhance
import numpy as np
from collections import defaultdict
import json

class MedicalDiffusionDatasetOrganizer:
    """
    Organizador de dataset médico que combina imagens originais e geradas por difusão,
    aplica pré-processamento e divide em treino/teste/validação balanceada.
    """
    def __init__(
        self,
        original_path: str,
        diffusion_generated_path: str,
        output_path: str,
        train_ratio: float = 0.7,
        val_ratio: float = 0.15,
        test_ratio: float = 0.15,
        target_size=(224, 224),
        apply_augmentation: bool = True
    ):
        self.original_path = Path(original_path)
        self.diffusion_generated_path = Path(diffusion_generated_path)
        self.output_path = Path(output_path)
        if abs(train_ratio + val_ratio + test_ratio - 1.0) > 0.001:
            raise ValueError("As proporções devem somar 1.0")
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.test_ratio = test_ratio
        self.target_size = target_size
        self.apply_augmentation = apply_augmentation
        self.classes = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
        self.stats = defaultdict(dict)
        random.seed(42)
        np.random.seed(42)

    def preprocess_image(self, image_path: Path, output_path: Path, 
                        apply_augmentation: bool = False, aug_suffix: str = "") -> bool:
        try:
            image = Image.open(image_path)
            if image.mode != 'RGB':
                if image.mode == 'RGBA':
                    background = Image.new('RGB', image.size, (255, 255, 255))
                    background.paste(image, mask=image.split()[-1])
                    image = background
                else:
                    image = image.convert('RGB')
            image = self._resize_with_padding(image, self.target_size)
            image = self._normalize_image(image)
            if apply_augmentation:
                image = self._apply_augmentation(image)
            output_file = output_path / f"{image_path.stem}{aug_suffix}.jpg"
            image.save(output_file, 'JPEG', quality=95)
            return True
        except Exception as e:
            print(f"Erro ao processar {image_path}: {str(e)}")
            return False

    def _resize_with_padding(self, image: Image.Image, target_size) -> Image.Image:
        target_w, target_h = target_size
        img_w, img_h = image.size
        ratio = min(target_w / img_w, target_h / img_h)
        new_w, new_h = int(img_w * ratio), int(img_h * ratio)
        image = image.resize((new_w, new_h), Image.Resampling.LANCZOS)
        if new_w != target_w or new_h != target_h:
            new_image = Image.new('RGB', target_size, (0, 0, 0))
            paste_x = (target_w - new_w) // 2
            paste_y = (target_h - new_h) // 2
            new_image.paste(image, (paste_x, paste_y))
            image = new_image
        return image

    def _normalize_image(self, image: Image.Image) -> Image.Image:
        enhancer = ImageEnhance.Contrast(image)
        image = enhancer.enhance(1.1)
        enhancer = ImageEnhance.Sharpness(image)
        image = enhancer.enhance(1.05)
        return image

    def _apply_augmentation(self, image: Image.Image) -> Image.Image:
        if random.random() > 0.5:
            angle = random.uniform(-5, 5)
            image = image.rotate(angle, fillcolor=(0, 0, 0))
        if random.random() > 0.5:
            factor = random.uniform(0.9, 1.1)
            enhancer = ImageEnhance.Brightness(image)
            image = enhancer.enhance(factor)
        if random.random() > 0.5:
            factor = random.uniform(0.9, 1.1)
            enhancer = ImageEnhance.Contrast(image)
            image = enhancer.enhance(factor)
        if random.random() > 0.7:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
        return image

    def collect_all_images(self):
        all_images = defaultdict(list)
        # Imagens originais
        for class_name in self.classes:
            original_class_path = self.original_path / class_name
            if original_class_path.exists():
                for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff']:
                    all_images[class_name].extend(original_class_path.glob(ext))
                    all_images[class_name].extend(original_class_path.glob(ext.upper()))
        # Imagens geradas por difusão
        for class_name in self.classes:
            gen_class_path = self.diffusion_generated_path / f"diffusion_generated_{class_name}"
            if gen_class_path.exists():
                for ext in ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff']:
                    all_images[class_name].extend(gen_class_path.glob(ext))
                    all_images[class_name].extend(gen_class_path.glob(ext.upper()))
        for class_name in all_images:
            all_images[class_name] = list(set(all_images[class_name]))
            random.shuffle(all_images[class_name])
        return all_images

    def split_data(self, all_images):
        splits = {'train': defaultdict(list), 'val': defaultdict(list), 'test': defaultdict(list)}
        for class_name, images in all_images.items():
            n_images = len(images)
            if n_images == 0:
                print(f"Aviso: Nenhuma imagem encontrada para classe {class_name}")
                continue
            n_train = int(n_images * self.train_ratio)
            n_val = int(n_images * self.val_ratio)
            n_test = n_images - n_train - n_val
            train_images = images[:n_train]
            val_images = images[n_train:n_train + n_val]
            test_images = images[n_train + n_val:]
            splits['train'][class_name] = train_images
            splits['val'][class_name] = val_images
            splits['test'][class_name] = test_images
            self.stats[class_name] = {
                'total': n_images,
                'train': len(train_images),
                'val': len(val_images),
                'test': len(test_images)
            }
        return splits

    def organize_dataset(self, max_augmentation_per_class=1000):
        print("🏥 Iniciando organização do dataset médico...")
        print(f"📁 Caminho original: {self.original_path}")
        print(f"🤖 Caminho gerado por difusão: {self.diffusion_generated_path}")
        print(f"📂 Caminho de saída: {self.output_path}")
        print(f"📊 Divisão: {self.train_ratio:.0%} treino, {self.val_ratio:.0%} validação, {self.test_ratio:.0%} teste")
        self._create_directory_structure()
        print("\n📋 Coletando imagens...")
        all_images = self.collect_all_images()
        self._print_initial_stats(all_images)
        print("\n🔄 Dividindo dados...")
        splits = self.split_data(all_images)
        for split_name, split_data in splits.items():
            print(f"\n📝 Processando {split_name}...")
            self._process_split(split_name, split_data, max_augmentation_per_class)
        self._save_final_stats()
        print("\n✅ Dataset organizado com sucesso!")
        print(f"📊 Estatísticas salvas em: {self.output_path / 'dataset_stats.json'}")

    def _create_directory_structure(self):
        self.output_path.mkdir(parents=True, exist_ok=True)
        for split in ['train', 'val', 'test']:
            for class_name in self.classes:
                (self.output_path / split / class_name).mkdir(parents=True, exist_ok=True)

    def _print_initial_stats(self, all_images):
        print("\n📊 Estatísticas iniciais:")
        total_images = 0
        for class_name in self.classes:
            count = len(all_images[class_name])
            total_images += count
            print(f"  {class_name}: {count:,} imagens")
        print(f"  Total: {total_images:,} imagens")

    def _process_split(self, split_name, split_data, max_augmentation):
        processed_count = 0
        augmented_count = 0
        for class_name, images in split_data.items():
            class_output_path = self.output_path / split_name / class_name
            for i, image_path in enumerate(images):
                success = self.preprocess_image(image_path, class_output_path)
                if success:
                    processed_count += 1
                if (i + 1) % 100 == 0:
                    print(f"    {class_name}: {i + 1}/{len(images)} processadas")
            if split_name == 'train' and self.apply_augmentation:
                n_augment = min(len(images), max_augmentation)
                augment_images = random.sample(images, n_augment)
                for i, image_path in enumerate(augment_images):
                    success = self.preprocess_image(
                        image_path, class_output_path, 
                        apply_augmentation=True, 
                        aug_suffix=f"_aug_{i}"
                    )
                    if success:
                        augmented_count += 1
                print(f"    {class_name}: {n_augment} imagens aumentadas")
        print(f"  ✅ {split_name}: {processed_count} processadas, {augmented_count} aumentadas")

    def _save_final_stats(self):
        final_stats = {
            'dataset_info': {
                'total_classes': len(self.classes),
                'classes': self.classes,
                'target_size': self.target_size,
                'splits': {
                    'train_ratio': self.train_ratio,
                    'val_ratio': self.val_ratio,
                    'test_ratio': self.test_ratio
                }
            },
            'class_distribution': {}
        }
        for split in ['train', 'val', 'test']:
            for class_name in self.classes:
                class_path = self.output_path / split / class_name
                if class_path.exists():
                    count = len(list(class_path.glob('*.jpg')))
                    if class_name not in final_stats['class_distribution']:
                        final_stats['class_distribution'][class_name] = {}
                    final_stats['class_distribution'][class_name][split] = count
        for class_name in self.classes:
            class_data = final_stats['class_distribution'][class_name]
            class_data['total'] = sum(class_data.values())
        stats_file = self.output_path / 'dataset_stats.json'
        with open(stats_file, 'w', encoding='utf-8') as f:
            json.dump(final_stats, f, indent=2, ensure_ascii=False)
        print("\n📊 Resumo Final:")
        for class_name in self.classes:
            class_data = final_stats['class_distribution'][class_name]
            print(f"  {class_name}:")
            print(f"    Treino: {class_data['train']:,}")
            print(f"    Validação: {class_data['val']:,}")
            print(f"    Teste: {class_data['test']:,}")
            print(f"    Total: {class_data['total']:,}")

    def verify_dataset(self):
        print("\n🔍 Verificando integridade do dataset...")
        issues = []
        for split in ['train', 'val', 'test']:
            for class_name in self.classes:
                class_path = self.output_path / split / class_name
                if not class_path.exists():
                    issues.append(f"Diretório ausente: {class_path}")
                    continue
                images = list(class_path.glob('*.jpg'))
                if len(images) == 0:
                    issues.append(f"Nenhuma imagem em: {class_path}")
                sample_images = random.sample(images, min(5, len(images)))
                for img_path in sample_images:
                    try:
                        with Image.open(img_path) as img:
                            if img.size != self.target_size:
                                issues.append(f"Tamanho incorreto: {img_path} ({img.size})")
                    except Exception as e:
                        issues.append(f"Imagem corrompida: {img_path} - {str(e)}")
        if issues:
            print("❌ Problemas encontrados:")
            for issue in issues:
                print(f"  - {issue}")
        else:
            print("✅ Dataset verificado com sucesso!")
        return len(issues) == 0

def main():
    ORIGINAL_PATH = "./dataSetPibic/train"  # Imagens originais
    DIFFUSION_GENERATED_PATH = "./"         # Pastas diffusion_generated_[classe]
    OUTPUT_PATH = "./dataset_organizado"
    TARGET_SIZE = (224, 224)
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15

    organizer = MedicalDiffusionDatasetOrganizer(
        original_path=ORIGINAL_PATH,
        diffusion_generated_path=DIFFUSION_GENERATED_PATH,
        output_path=OUTPUT_PATH,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        target_size=TARGET_SIZE,
        apply_augmentation=True
    )

    try:
        organizer.organize_dataset(max_augmentation_per_class=500)
        organizer.verify_dataset()
        print("\n🎉 Processo concluído com sucesso!")
        print(f"📂 Dataset organizado disponível em: {OUTPUT_PATH}")
    except Exception as e:
        print(f"❌ Erro durante a organização: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()


🏥 Iniciando organização do dataset médico...
📁 Caminho original: dataSetPibic/train
🤖 Caminho gerado por difusão: .
📂 Caminho de saída: dataset_organizado
📊 Divisão: 70% treino, 15% validação, 15% teste

📋 Coletando imagens...

📊 Estatísticas iniciais:
  covid19: 15,710 imagens
  normal: 15,766 imagens
  pneumonia_bacterial: 15,740 imagens
  pneumonia_viral: 15,678 imagens
  Total: 62,894 imagens

🔄 Dividindo dados...

📝 Processando train...
    covid19: 100/10997 processadas
    covid19: 200/10997 processadas
    covid19: 300/10997 processadas
    covid19: 400/10997 processadas
    covid19: 500/10997 processadas
    covid19: 600/10997 processadas
    covid19: 700/10997 processadas
    covid19: 800/10997 processadas
    covid19: 900/10997 processadas
    covid19: 1000/10997 processadas
    covid19: 1100/10997 processadas
    covid19: 1200/10997 processadas
    covid19: 1300/10997 processadas
    covid19: 1400/10997 processadas
    covid19: 1500/10997 processadas
    covid19: 1600/10997